# CS3255 NLP Lab Assignment 7: Demystifying Word Embeddings (Word2Vec to FastText)

## Aim
To train and evaluate Word2Vec, GloVe, and FastText word embeddings, and to compare pre-trained GloVe-Twitter vectors with FastText vectors as features for classifying tech-support customer tickets as resolved or frustrated.

## Problem Statement
- **Parts A-C:** Train a Skip-gram Word2Vec model on a toy corpus, solve the semantic analogy *king - man + woman = queen* using pre-trained GloVe (`glove-wiki-gigaword-50`), and generate embeddings for out-of-vocabulary (OOV) words using subword representations in FastText.
- **Part D (Assignment):** Create 10 resolved and 10 frustrated customer support statements. Represent each sentence as the mean vector of its constituent tokens using (A) pre-trained `glove-twitter-25` and (B) a custom FastText model trained on the domain dataset. Train a Logistic Regression classifier on each representation and evaluate on a held-out tricky support tweet.

## Program

In [1]:
# Install required packages for Google Colab environment
# !pip install -q gensim scikit-learn numpy

import re
import numpy as np
import gensim.downloader as api
from gensim.models import Word2Vec, FastText
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def tokenize(text):
    return re.sub(r'[^a-z\s]', '', text.lower()).split()

### Part A: Word2Vec (Skip-gram)

In [2]:
corpus = ["the quick brown fox jumps over the lazy dog",
          "the dog barked at the mailman",
          "the cat sat on the mat",
          "dogs and cats are popular pets"]
tokenized_corpus = [tokenize(s) for s in corpus]

w2v = Word2Vec(sentences=tokenized_corpus, vector_size=10, window=2,
               min_count=1, sg=1, seed=42, workers=1)
print("Word2Vec vector for 'dog':\n", np.round(w2v.wv["dog"], 4))
print("Words most similar to 'dog':")
for w, s in w2v.wv.most_similar("dog", topn=3):
    print(f"  - {w} (score: {s:.4f})")

Word2Vec vector for 'dog':
 [ 0.0054  0.0952  0.0473  0.0522  0.0439  0.0571  0.0026 -0.0748  0.068
 -0.0098]
Words most similar to 'dog':
  - on (score: 0.3038)
  - cat (score: 0.2956)
  - jumps (score: 0.2776)


### Part B: Pre-trained GloVe

In [3]:
glove_wiki = api.load("glove-wiki-gigaword-50")
ans = glove_wiki.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
print(f"\nGloVe analogy  king - man + woman = {ans[0][0]} ({ans[0][1]:.4f})")
print(f"Similarity(apple, banana) = {glove_wiki.similarity('apple', 'banana'):.4f}")
print(f"Similarity(apple, car)    = {glove_wiki.similarity('apple', 'car'):.4f}")


GloVe analogy  king - man + woman = queen (0.8524)
Similarity(apple, banana) = 0.5608
Similarity(apple, car)    = 0.3952


### Part C: FastText and OOV words

In [4]:
ft = FastText(sentences=tokenized_corpus, vector_size=10, window=2,
              min_count=1, seed=42, workers=1)
oov = "doggy"
print(f"\n'{oov}' in Word2Vec vocabulary? {oov in w2v.wv.key_to_index}")
print(f"FastText vector for OOV word '{oov}':\n", np.round(ft.wv[oov], 4))
print(f"Words similar to '{oov}':",
      [(w, round(s, 4)) for w, s in ft.wv.most_similar(oov, topn=3)])


'doggy' in Word2Vec vocabulary? False
FastText vector for OOV word 'doggy':
 [-0.0197 -0.0039  0.0098 -0.0118 -0.0058  0.0049  0.0221  0.0126  0.0076
 -0.0044]
Words similar to 'doggy': [('cat', 0.6291), ('on', 0.5483), ('mailman', 0.4667)]


### Part D: Assignment - compare embeddings for sentiment

In [5]:
resolved = [
    "The new patch fixed my router, thanks!",
    "Customer service was incredibly helpful.",
    "My issue was resolved in just five minutes.",
    "Great support, the technician solved everything quickly.",
    "The update works perfectly now, thank you so much.",
    "Thanks for the quick response, my laptop is working again.",
    "The agent was polite and fixed my billing problem.",
    "Internet speed is back to normal, excellent service.",
    "I am happy with how fast you replaced my charger.",
    "Problem solved, the app runs smoothly now.",
]
frustrated = [
    "My software keeps crashing after the update.",
    "I have been on hold for 45 minutes!",
    "Nobody has replied to my ticket in three days.",
    "The router stopped working again and support is useless.",
    "I am so annoyed, the app freezes every time I open it.",
    "Worst customer service ever, my problem is still not fixed.",
    "The laptop overheats and you keep ignoring my emails.",
    "Why is my internet so slow, this is terrible.",
    "I was charged twice and still have no refund.",
    "The new update broke everything, very disappointed.",
]
texts = resolved + frustrated
y = np.array([1] * 10 + [0] * 10)          # 1 = resolved, 0 = frustrated
tokens = [tokenize(t) for t in texts]

# Model A: pre-trained GloVe (Twitter, 25-d)
glove_tw = api.load("glove-twitter-25")
# Model B: FastText trained from scratch on the support dataset
ft_support = FastText(sentences=tokens, vector_size=25, window=3, min_count=1,
                      epochs=50, seed=42, workers=1)

def sentence_vector(words, kv, dim):
    """Average word vectors; skip OOV words (zero vector if none found)."""
    vecs = [kv[w] for w in words if w in kv.key_to_index]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

def ft_sentence_vector(words, model):
    return np.mean([model.wv[w] for w in words], axis=0)   # FastText handles OOV

oov_count = sum(w not in glove_tw.key_to_index for t in tokens for w in t)
print(f"\nGloVe-Twitter OOV tokens in dataset: {oov_count} / {sum(map(len, tokens))}")

X_glove = np.array([sentence_vector(t, glove_tw, 25) for t in tokens])
X_ft = np.array([ft_sentence_vector(t, ft_support) for t in tokens])

clf_glove = LogisticRegression(max_iter=1000).fit(X_glove, y)
clf_ft = LogisticRegression(max_iter=1000).fit(X_ft, y)

cv_glove = cross_val_score(LogisticRegression(max_iter=1000), X_glove, y, cv=5).mean()
cv_ft = cross_val_score(LogisticRegression(max_iter=1000), X_ft, y, cv=5).mean()
print("\n5-fold cross-validation accuracy:")
print(f"  GloVe-Twitter + LogReg : {cv_glove:.2f}")
print(f"  FastText      + LogReg : {cv_ft:.2f}")

label = {1: "Resolved", 0: "Frustrated"}
tests = ["The wifi is behaving weirdly today, what is going on?",
         "Thanks a lot, everything works fine now!"]
print("\nPredictions on new support tweets:")
for t in tests:
    tk = tokenize(t)
    pg = clf_glove.predict_proba([sentence_vector(tk, glove_tw, 25)])[0]
    pf = clf_ft.predict_proba([ft_sentence_vector(tk, ft_support)])[0]
    print(f"\n  Tweet: {t}")
    print(f"  GloVe-Twitter -> {label[int(pg.argmax())]:<10} (P(resolved) = {pg[1]:.2f})")
    print(f"  FastText      -> {label[int(pf.argmax())]:<10} (P(resolved) = {pf[1]:.2f})")


GloVe-Twitter OOV tokens in dataset: 0 / 168

5-fold cross-validation accuracy:
  GloVe-Twitter + LogReg : 0.70
  FastText      + LogReg : 0.50

Predictions on new support tweets:

  Tweet: The wifi is behaving weirdly today, what is going on?
  GloVe-Twitter -> Frustrated (P(resolved) = 0.48)
  FastText      -> Frustrated (P(resolved) = 0.50)

  Tweet: Thanks a lot, everything works fine now!
  GloVe-Twitter -> Resolved   (P(resolved) = 0.56)
  FastText      -> Frustrated (P(resolved) = 0.50)


## Observations
- The toy Word2Vec model's neighbours (*on*, *cat*) are noisy: four sentences are far too little data for meaningful vectors.
- Pre-trained GloVe solves king − man + woman = **queen** (0.85) and rates apple–banana (0.56) as more similar than apple–car (0.40), showing that it captures semantic relations learned from a large corpus.
- FastText built a vector for the unseen word *doggy* from its character n-grams; Word2Vec has no entry for it and would raise a KeyError.
- GloVe-Twitter features reach 0.70 cross-validation accuracy, while FastText trained on just 20 sentences reaches 0.50 (chance level): its sentence vectors are nearly identical, so it predicts ≈ 0.50 for every input. Pre-training on billions of tweets gives GloVe-Twitter useful knowledge of words like *thanks*, *annoyed* and *terrible*, with no OOV words in this dataset.
- For the tricky tweet, GloVe-Twitter leans *Frustrated* with low confidence (P(resolved) = 0.48), which suits a message reporting an unexplained problem, while FastText gives no information (0.50). GloVe-Twitter captured the nuance better. Text → tokens → embeddings → classifier works, but the quality of the embeddings decides the result.

## Result
Word2Vec, GloVe and FastText embeddings were trained or loaded and explored, and sentence vectors from GloVe-Twitter and FastText were compared for sentiment classification.